In [ ]:
SELECT table_name, column_name
FROM spark_catalog.silver.information_schema.columns
WHERE lower(column_name) LIKE '%service%'
   OR lower(column_name) LIKE '%clienttype%'
   OR lower(column_name) LIKE '%tenancy%'
ORDER BY table_name, column_name;

In [ ]:
SELECT DISTINCT trim(clienttype) AS v
FROM <that_table>
WHERE clienttype IS NOT NULL AND trim(clienttype) <> ''
LIMIT 50;

In [ ]:
# Search columns across all tables in a database (Lakehouse schema)
db = "silver"
patterns = ["service", "clienttype", "tenancy"]

tables = [t.name for t in spark.catalog.listTables(db) if t.tableType.lower() != "view"]
hits = []

for tbl in tables:
    cols = [c.name.lower() for c in spark.table(f"{db}.{tbl}").schema.fields]
    if any(any(p in col for p in patterns) for col in cols):
        for c in cols:
            if any(p in c for p in patterns):
                hits.append((tbl, c))

display(spark.createDataFrame(hits, ["table_name", "column_name"]).orderBy("table_name", "column_name"))

--------------new------------------

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH src AS (
    SELECT DISTINCT
        cp.z_src_system_instance                          AS service_src_sys_inst_id,
        trim(cp.cprod_service)                            AS service_src_name,
        lower(trim(cp.cprod_service))                     AS service_src_name_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'
),
src_with_id AS (
    SELECT
        service_src_sys_inst_id,
        service_src_name,
        concat(service_src_sys_inst_id, '|', service_src_name_norm) AS service_src_id_norm
    FROM src
),
rdm AS (
    SELECT DISTINCT
        lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)
SELECT
    -- 3 mandatory fields (per Mali)
    s.service_src_id_norm     AS service_src_id,
    s.service_src_name        AS service_src_name,
    s.service_src_sys_inst_id AS service_src_sys_inst_id,

    -- useful audit
    current_timestamp()       AS z_src_created_date_time,
    current_user()            AS z_src_created_by_user,
    current_timestamp()       AS z_src_modified_date_time,
    current_user()            AS z_src_modified_by_user,

    1                         AS z_order_is_active
FROM src_with_id s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT count(*) AS new_rows FROM silver.silver_rdm_service_add;

%sql
SELECT service_src_sys_inst_id, count(*) AS c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

%sql
SELECT * FROM silver.silver_rdm_service_add LIMIT 50;

In [ ]:
आपण कोणते tables वापरले?
1) Source table (transactional-ish / curated)

silver.silver_care_product
इथून आपण “systems मध्ये दिसणारे services” (आपल्या case मध्ये cprod_service) काढले.

2) Reference table (SharePoint ingested RDM)

silver.silver_rdm_service
इथे आधीपासून जे service IDs SharePoint मध्ये आहेत ते आहेत, त्यामुळे “नवीन” काय आहे ते ठरवायला याचा वापर.

3) Output table

silver.silver_rdm_service_add
हा आपला “candidates/additions” टेबल, जो पुढे semantic model + Power Automate ने SharePoint मध्ये push करतो.

Code मध्ये CTE-by-CTE काय केलं?
CTE 1: src

उद्देश: source system मधून candidate services काढणे (duplicate कमी करणे)

From: silver.silver_care_product cp

Columns घेतले:

cp.z_src_system_instance → नाव दिलं: service_src_sys_inst_id
(हे “instance” आहे: IAPT 427, WIP001, SONE Y06008 वगैरे)

cp.cprod_service → trim करून नाव दिलं: service_src_name
(हे source system मधला service label)

cp.cprod_service → lower+trim करून नाव दिलं: service_src_name_norm
(matching साठी normalized version)

Filters:

cprod_service IS NOT NULL

trim(cprod_service) <> ''

lower(trim(cprod_service)) <> 'unknown'
(Unknown candidates push करायचे नाहीत म्हणून)

SELECT DISTINCT का?
एकाच service value हजारो rows मध्ये repeat असू शकते. त्यामुळे unique candidate set तयार केला.

CTE 2: src_with_id

उद्देश: Mali ने सांगितल्याप्रमाणे unique service_src_id बनवणे.

From: src

काय केलं:

service_src_id_norm = concat(service_src_sys_inst_id, '|', service_src_name_norm)

म्हणजे ID = instance + “|” + normalized name
उदा:

IAPT 427|intervention given

WIP001|helpline

SONE Y06008|rheumatology

हे का? (grain rule)
Mali म्हणतो: टेबलचा grain = instance by service.
त्यामुळे instance + name combine केलं की unique ID तयार होतो.

CTE 3: rdm

उद्देश: SharePoint/RDM मध्ये आधीपासून असलेले IDs काढणे, जेणेकरून duplicates टाळता येतील.

From: silver.silver_rdm_service

Columns घेतले:

service_src_id → lower+trim करून नाव दिलं: service_src_id_norm

Filter:

service_src_id IS NOT NULL

trim(service_src_id) <> ''

इथे आपण service_id (SharePoint generated) वापरला नाही, कारण तो source systems शी match होत नाही.

Final SELECT मध्ये काय केलं?
Join logic (मुख्य match)

Left join: src_with_id s LEFT JOIN rdm r

Join condition:

s.service_src_id_norm = r.service_src_id_norm

Why left join?
आपल्याला source मधल्या सर्व candidates ठेवायचे आहेत आणि जे RDM मध्ये आधीच आहेत ते drop करायचे आहेत.

Final filter:

WHERE r.service_src_id_norm IS NULL

याचा अर्थ:
RDM मध्ये match नाही → हा new service आहे → _add मध्ये टाका.

Output table मध्ये कोणते columns populate झाले?

silver.silver_rdm_service_add मध्ये आपण टाकले:

Mali ने सांगितलेले mandatory 3 fields:

service_src_id ← s.service_src_id_norm

service_src_name ← s.service_src_name

service_src_sys_inst_id ← s.service_src_sys_inst_id

Audit/ops fields:

z_src_created_date_time = current_timestamp()

z_src_created_by_user = current_user()

z_src_modified_date_time = current_timestamp()

z_src_modified_by_user = current_user()

z_order_is_active = 1

हे पुढे “who/when inserted” trace करायला useful आहे.